# Readout Guardrails: R1 / R2 / R3

This notebook adds readout diagnostics to ATR without presupposing outcome.

- **R1**: confidence-aware readout fields (margin, entropy)
- **R2**: ID-first token trace (IDs + display strings)
- **R3**: tensor-readout concordance categories

Interpretation principle:

> If token labels flicker while `cos_sim_mean` is high and margin is low, treat this as readout ambiguity first.


In [1]:
# STEP 0: Imports and setup
# --- repo-root bootstrap: resolve imports + output paths from any launch dir ---
import os, sys
from pathlib import Path
_here = Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "atr_engine.py").exists():
        os.chdir(_cand)
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break
# --- end bootstrap ---

import json

import torch
from transformer_lens import HookedTransformer

from atr_engine import run_atr_loop

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


C:\Users\Fab2\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
# STEP 1: Configuration (neutral labels)
MODEL_NAME = "gpt2-small"
PROMPT = "The cat sat on the mat and then"
LAYER_START = 0
LAYER_END = 11
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# Concordance thresholds for R3 categorisation
HIGH_COS_THRESHOLD = 0.995
LOW_MARGIN_THRESHOLD = 0.2

OUTPUT_PATH = Path("experiments/output/readout_guardrails_gpt2_small.json")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Layers:", LAYER_START, "->", LAYER_END)
print("Schedule:", ITERATION_SCHEDULE)

Model: gpt2-small
Layers: 0 -> 11
Schedule: [0, 2, 3, 5, 10, 20, 50, 100]


In [3]:
# STEP 2: Run ATR loop with confidence-aware readout from atr_engine.py
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()

with torch.no_grad():
    snapshots = run_atr_loop(
        model=model,
        prompt=PROMPT,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE,
        verbose=True,
    )

print(f"Snapshots captured: {len(snapshots)}")

C:\Users\Fab2\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Fab2\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Loaded pretrained model gpt2-small into HookedTransformer


  iter   2: top='the', cos_mean=0.7830, pos_collapse=0.9596
  iter   3: top='the', cos_mean=-0.7841, pos_collapse=0.9464


  iter   5: top='fem', cos_mean=-0.1679, pos_collapse=0.9930


  iter  10: top='Ag', cos_mean=0.6049, pos_collapse=0.9993


  iter  20: top='Zero', cos_mean=0.9732, pos_collapse=1.0000


  iter  50: top='Divine', cos_mean=0.6701, pos_collapse=1.0000


  iter 100: top='Divine', cos_mean=0.6776, pos_collapse=1.0000
Snapshots captured: 8


In [4]:
# STEP 3: R3 concordance classification (neutral categories)
def classify_snapshot(cos_sim_mean, top_logit_margin, high_cos, low_margin):
    if cos_sim_mean >= high_cos and top_logit_margin <= low_margin:
        return "high_cos_low_margin"
    if cos_sim_mean >= high_cos and top_logit_margin > low_margin:
        return "high_cos_high_margin"
    return "lower_cos"

summary_rows = []
category_counts = {
    "high_cos_low_margin": 0,
    "high_cos_high_margin": 0,
    "lower_cos": 0,
}

for s in snapshots:
    category = classify_snapshot(
        cos_sim_mean=float(s["cosine_sim_mean"]),
        top_logit_margin=float(s["top_logit_margin_last"]),
        high_cos=HIGH_COS_THRESHOLD,
        low_margin=LOW_MARGIN_THRESHOLD,
    )
    category_counts[category] += 1

    summary_rows.append({
        "iteration": int(s["iteration"]),
        "cosine_sim_mean": float(s["cosine_sim_mean"]),
        "cosine_sim_last": float(s["cosine_sim_last"]),
        "position_similarity": float(s["position_similarity"]),
        "top_token_id": int(s["top_token_ids_last"][0]),
        "top_token_string": str(s["top_token_strings_last"][0]),
        "top_token_prob": float(s["top_token_probs_last"][0]),
        "top_logit_margin": float(s["top_logit_margin_last"]),
        "entropy": float(s["entropy_last"]),
        "all_position_token_ids": [int(x) for x in s["all_position_token_ids"]],
        "all_position_token_strings": [str(x) for x in s["all_position_token_strings"]],
        "concordance_category": category,
    })

print("Category counts:", category_counts)

Category counts: {'high_cos_low_margin': 1, 'high_cos_high_margin': 0, 'lower_cos': 7}


In [5]:
# STEP 4: Save machine-readable output for follow-on analysis
payload = {
    "config": {
        "model_name": MODEL_NAME,
        "prompt": PROMPT,
        "layer_start": LAYER_START,
        "layer_end": LAYER_END,
        "max_iter": MAX_ITERATIONS,
        "schedule": ITERATION_SCHEDULE,
        "high_cos_threshold": HIGH_COS_THRESHOLD,
        "low_margin_threshold": LOW_MARGIN_THRESHOLD,
    },
    "category_counts": category_counts,
    "rows": summary_rows,
}

OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}")

Saved: experiments\output\readout_guardrails_gpt2_small.json
